# Learning Objectives

In this notebook, you will 
- learn the concept of ETL
- write ETL jobs for CSV files from `pgexercises` https://pgexercises.com/gettingstarted.html

# What's ETL or ELT?

ETL stands for Extract, Transform, Load. In the context of Spark, ETL refers to the process of extracting data from various sources, transforming it into a desired format or structure, and loading it into a target system, such as a data warehouse or a data lake.

Here's a breakdown of each step in the ETL process:

## Extract
This step involves extracting data from multiple sources, such as databases, files (CSV, JSON, Parquet), APIs, or streaming data sources. Spark provides connectors and APIs to read data from a wide range of sources, allowing you to extract data in parallel and efficiently handle large datasets.

## Transform
In the transform step, the extracted data is processed and transformed according to specific business logic or requirements. This may involve cleaning the data, applying calculations or aggregations, performing data enrichment, filtering, joining datasets, or any other data manipulation operations. Spark provides a powerful set of transformation functions and SQL capabilities to perform these operations efficiently in a distributed and scalable manner.

## Load
Once the data has been transformed, it is loaded into a target system, such as a data warehouse, a data lake, or another storage system. Spark allows you to write the transformed data to various output formats and storage systems, including databases, distributed file systems (like Hadoop Distributed File System or Amazon S3), or columnar formats like Delta Lake or Apache Parquet. The data can be partitioned, sorted, or structured to optimize querying and analysis.

Spark's distributed computing capabilities, scalability, and rich ecosystem of libraries make it a popular choice for ETL workflows. It can handle large-scale data processing, perform complex transformations, and efficiently load data into different target systems.

By leveraging Spark for ETL, organizations can extract data from diverse sources, apply transformations to ensure data quality and consistency, and load the transformed data into a central repository for further analysis, reporting, or machine learning tasks.

# Enable DBFS UI

- Setting -> Admin Console -> search for dbfs

<img src="https://raw.githubusercontent.com/jarviscanada/jarvis_data_eng_demo/feature/data/spark/notebook/spark_fundamentals/img/entable_dbfs.jpg" width="700">

- Refresh the page and view DBFS files from UI

<img src="https://raw.githubusercontent.com/jarviscanada/jarvis_data_eng_demo/feature/data/spark/notebook/spark_fundamentals/img/dbfs%20ui.png" width="700">

## Import `pgexercises` CSV files

- The pgexercises CSV data files can be found [here](https://github.com/jarviscanada/jarvis_data_eng_demo/tree/feature/data/spark/data/pgexercises).
- The pgexercises schema can be found [here](https://pgexercises.com/gettingstarted.html) (for reference purposes).
- Upload the `bookings.csv`, `facilities.csv`, and `members.csv` files using Databricks UI (see screenshot)
- You can view the imported files from the DBFS UI.

![Upload Files](https://raw.githubusercontent.com/jarviscanada/jarvis_data_eng_demo/feature/data/spark/notebook/spark_fundamentals/img/upload%20file.png)

# Interview Questions

While completing the rest of the practice, try to answer the following questions:

## Concepts
- What is ETL? (Hint: Explain each step)

## Databricks
- What is Databricks?
  - Databricks is a unified, open analytics platform for building, deploying, sharing, and maintaining enterprise-grade data, analytics, and AI solutions at scale.
- What is a Notebook?
  - 
- What is DBFS?
- What is a cluster? 
- Is Databricks a data lake or a data warehouse?

## Managed Table
- What is a managed table in Databricks?
- Can you explain how to create a managed table in Databricks?
- Can you compare a managed table with an RDBMS table? (Hint: Schema on read vs schema on write)
- What is the Hive metastore and how does it relate to managed tables in Databricks?
- How does a managed table differ from an unmanaged (external) table in Databricks? (Hint: Consider what happens to the data when the table is deleted)
- How can you define a schema for a managed table?

## Spark
`df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_location)`
- What does the option("inferSchema", "true") do? 
- What does the option("header", "true") do?
- How can you write data to a managed table?
- How can you read data from a managed table into a DataFrame?

# ETL `bookings.csv` file

- **Extract**: Load data from CSV file into a DF
- **Transformation**: no transformation needed as we want to load data as it
- **Load**: Save the DF into a managed table (or Hive table); 

# Managed Table
This is an important interview topic. Some people may refer to managed tables as Hive tables.

https://docs.databricks.com/data-governance/unity-catalog/create-tables.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType

file_location = "/FileStore/tables/bookings.csv"

# What does `option("header", "true")` and `option("inferSchema", "true")` do?
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_location)

# Why the df schema doesn't match the DDL data type? https://pgexercises.com/gettingstarted.html (hint: `option("inferSchema", "true")`)
df.printSchema()

df.write.saveAsTable("bookings1")

# Here is the solution to define schema manually
# Define schema for the bookings table
schema = StructType([
    StructField("bookid", IntegerType(), True),
    StructField("facid", IntegerType(), True),
    StructField("memid", IntegerType(), True),
    StructField("starttime", TimestampType(), True),
    StructField("slots", IntegerType(), True)
])

# Read data from CSV file into DataFrame with predefined schema
df = spark.read.format("csv").option("header", "true").schema(schema).load(file_location)

# No 

# Drop the table if it already exists
spark.sql("DROP TABLE IF EXISTS bookings")

# Write data from DataFrame into managed table
df.write.saveAsTable("bookings1")

#df.write.saveAsTable("bookings")
print("bookings")


root
 |-- bookid: integer (nullable = true)
 |-- facid: integer (nullable = true)
 |-- memid: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- slots: integer (nullable = true)

bookings


# Complete ETL Jobs

- Complete ETL for `facilities.csv` and `members.csv`
- Tips
  - The Databricks community version will terminate the cluster after a few hours of inactivity. As a result, all managed tables will be deleted. You will need to rerun this notebook to perform the ETL on all files for the other exercises.
  - DBFS data will not be deleted when a custer become inactive/deleted

In [0]:
# Write a ETL job for `facilities.csv`
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

file_location = "/FileStore/tables/facilities.csv"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_location)

df.printSchema()

schema = StructType([
    StructField("facid", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("membercost", DoubleType(), True),
    StructField("guestcost", DoubleType(), True),
    StructField("initialoutlay", DoubleType(), True),
    StructField("monthlymaintenance", DoubleType(), True)
])

df = spark.read.format("csv").option("header", "true").schema(schema).load(file_location)

spark.sql("DROP TABLE IF EXISTS facilities")

df.write.saveAsTable("facilities")



root
 |-- facid: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- membercost: double (nullable = true)
 |-- guestcost: double (nullable = true)
 |-- initialoutlay: integer (nullable = true)
 |-- monthlymaintenance: integer (nullable = true)



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-3076586410273618>:23
     19 df = spark.read.format("csv").option("header", "true").schema(schema).load(file_location)
     21 spark.sql("DROP TABLE IF EXISTS facilities")
---> 23 df.write.saveAsTable("facilities")

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1520, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
   1518 if format is not None:
   1519     self.format(format)
-> 1520 self._jwrite.saveAsTable(name)


In [0]:
# Write a ETL job Complete ETL for `members.csv`
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, StringType

file_location = "/FileStore/tables/members.csv"

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_location)

df.printSchema()

schema = StructType ([
    StructField("memid", IntegerType(), True),
    StructField("surname", StringType(), True),
    StructField("firstname", StringType(), True),
    StructField("address", StringType(), True),
    StructField("zipcode", IntegerType(), True),
    StructField("telephone", StringType(), True),
    StructField("recommendedby", IntegerType(), False),
    StructField("joindate", TimestampType(), True)
])

df = spark.read.format("csv").option("header", "true").schema(schema).load(file_location)

spark.sql("DROP TABLE IF EXISTS members")

df.write.saveAsTable("members")

# Save your work to Git

- Export the notebook to IPYTHON format, `notebook top menu bar -> File -> Export -> iphython`
- Upload to your Git repository, `your_repo/spark/notebooks/`
- Github can render ipython notebook https://github.com/josephcslater/JupyterExamples/blob/master/Calc_Review.ipynb